# Retail Sales Dashboard

A Jupyter version of `app_code.txt`. It reads the seven original CSVs in `Datasets/`, prepares sales data, and reproduces the app's KPIs, six charts, and transaction view.

**Run it in Jupyter.** Save this notebook in **Capstone Project**, one level above **Datasets**, then select **Run All**. Use a Python kernel with pandas 2.0 or later and Matplotlib. Edit the four filters in the filter cell and rerun from there to explore a segment.

Sales metrics use customers, products, orders and order items. Payments, shipments and returns are loaded and counted in the input summary. The source app does not use those three tables to calculate sales.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
import pandas as pd
from IPython.display import display

DATA_DIR = Path("Datasets")
CHARTS_DIR = Path("charts") / "retail_sales_dashboard"

TABLES = ("customers", "products", "orders", "order_items", "payments", "shipments", "returns")
paths = {name: DATA_DIR / f"{name}.csv" for name in TABLES}
missing = [str(path) for path in paths.values() if not path.is_file()]
if missing:
    raise FileNotFoundError(
        f"Current working directory: {Path.cwd()}\n"
        f"Missing raw CSVs: {', '.join(missing)}\n"
        "Open this notebook from Capstone Project or update DATA_DIR."
    )

id_columns = {column: "string" for column in ("customer_id", "product_id", "order_id")}
raw = {name: pd.read_csv(path, dtype=id_columns) for name, path in paths.items()}
display(pd.DataFrame({"Source file": [path.name for path in paths.values()],
                      "Raw rows": [len(table) for table in raw.values()]}))

## Prepare the sales data

Remove exact duplicates, standardize product categories, parse dates and prices, and retain only item lines that match the order and master tables. Sales include orders marked **Completed** or **Shipped** with a valid order date.

In [ ]:
def clean_money(series):
    return pd.to_numeric(
        series.astype("string").str.replace("₱", "", regex=False)
        .str.replace(",", "", regex=False).str.strip(),
        errors="coerce",
    )


customers, products, orders, items = (
    raw[name].drop_duplicates().copy()
    for name in ("customers", "products", "orders", "order_items")
)
for table, key in ((customers, "customer_id"), (products, "product_id"), (orders, "order_id")):
    table.dropna(subset=[key], inplace=True)
    if table[key].duplicated().any():
        raise ValueError(f"Resolve conflicting rows with the same {key} before joining.")

CATEGORY_MAP = {
    'electronics': 'Electronics',
    'elec': 'Electronics',
    'home & kitchen': 'Home & Kitchen',
    'home and kitchen': 'Home & Kitchen',
    'books': 'Books',
    'book': 'Books',
    'groceries': 'Groceries',
    'grocery': 'Groceries',
    'toys': 'Toys',
    'toy': 'Toys',
    'beauty': 'Beauty',
    'beauty & care': 'Beauty',
    'fashion': 'Fashion',
    'fashon': 'Fashion',
    'sports': 'Sports',
    'sports & outdoor': 'Sports',
}
products["category_clean"] = (
    products["category"].astype("string").str.strip().str.lower()
    .replace(CATEGORY_MAP).replace("", "Unknown").fillna("Unknown")
)
items["quantity"] = pd.to_numeric(items["quantity"], errors="coerce")
items["unit_price_clean"] = clean_money(items["unit_price_at_purchase"])
items = items.loc[items["quantity"].gt(0) & items["unit_price_clean"].notna()].copy()

orders["order_date"] = pd.to_datetime(orders["order_datetime"], format="mixed", errors="coerce")
discount = pd.to_numeric(orders["discount_pct"], errors="coerce")
orders["discount_pct"] = discount.where(discount.between(0, 1), 0)
orders = orders.loc[
    orders["customer_id"].isin(customers["customer_id"])
    & orders["order_status"].isin(["Completed", "Shipped"])
    & orders["order_date"].notna()
].copy()

sales = (
    items.merge(orders, on="order_id", how="inner", validate="many_to_one")
    .merge(products[["product_id", "product_name", "brand", "category_clean"]],
           on="product_id", how="inner", validate="many_to_one")
    .merge(customers[["customer_id", "region_name", "city", "loyalty_tier"]],
           on="customer_id", how="left", validate="many_to_one")
)
sales["gross_sales"] = sales["quantity"] * sales["unit_price_clean"]
sales["net_sales"] = sales["gross_sales"] * (1 - sales["discount_pct"])
sales["month"] = sales["order_date"].dt.to_period("M").astype("string")
print(f"Eligible item rows: {len(sales):,} | Eligible orders: {sales['order_id'].nunique():,}")

## Filter the dashboard

Use `"All"` to leave a filter open. Month uses `YYYY-MM`. The table below lists values from the data.

In [ ]:
filter_columns = {
    "Month": "month",
    "Region": "region_name",
    "Sales channel": "sales_channel",
    "Product category": "category_clean",
}
filter_options = {
    label: sorted(sales[column].dropna().astype(str).unique())
    for label, column in filter_columns.items()
}
display(pd.Series({label: ", ".join(values) for label, values in filter_options.items()},
                  name="Available values").to_frame())

In [ ]:
selected = {
    "Month": "All",
    "Region": "All",
    "Sales channel": "All",
    "Product category": "All",
}

filtered = sales.copy()
for label, value in selected.items():
    if value == "All":
        continue
    if value not in filter_options[label]:
        raise ValueError(f"{value!r} is not a valid {label}. See the options above.")
    filtered = filtered.loc[filtered[filter_columns[label]].eq(value)]

if filtered.empty:
    print("No transactions match these filters. Change a filter and rerun this cell.")

## Sales KPIs

Gross sales are quantity × purchase price. Net sales apply the order discount and exclude shipping fees and refunds. Average order value uses the sum of net sales per distinct order. With a category filter, it measures the average value of matching items per order.

In [ ]:
order_values = filtered.groupby("order_id")["net_sales"].sum()
category_sales = filtered.groupby("category_clean")["gross_sales"].sum()
aov = f"₱{order_values.mean():,.2f}" if not order_values.empty else "—"

kpis = pd.Series({
    "Gross sales": f"₱{filtered['gross_sales'].sum():,.2f}",
    "Net sales": f"₱{filtered['net_sales'].sum():,.2f}",
    "Unique orders": f"{len(order_values):,}",
    "Average order value": aov,
    "Top category by gross sales": category_sales.idxmax() if not category_sales.empty else "—",
}, name="Value")
display(kpis.to_frame())

## Sales charts

Each figure appears below its code and saves a PNG to `charts/retail_sales_dashboard/`. Running the charts again updates the files with the current filter results. Months without matching sales are shown as gaps.

In [ ]:
BLUE = "#256d85"
TEAL = "#118577"


def finish_chart(fig, ax, filename, currency_axis="x"):
    axis = ax.xaxis if currency_axis == "x" else ax.yaxis
    axis.set_major_formatter(StrMethodFormatter("₱{x:,.0f}"))
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis=currency_axis, alpha=0.22)
    ax.set_axisbelow(True)
    CHARTS_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(CHARTS_DIR / filename, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def sales_bars(values, title, label, filename, horizontal=True):
    values = values.sort_values(ascending=horizontal)
    fig, ax = plt.subplots(figsize=(10, max(3.6, 0.34 * len(values) + 1.3) if horizontal else 4),
                           layout="constrained")
    if horizontal:
        ax.barh(values.index.astype(str), values, color=TEAL)
        ax.set(xlabel="Gross sales (PHP)", ylabel=label, title=title)
    else:
        ax.bar(values.index.astype(str), values, color=BLUE)
        ax.set(xlabel=label, ylabel="Gross sales (PHP)", title=title)
        ax.tick_params(axis="x", labelrotation=20)
    finish_chart(fig, ax, filename, "x" if horizontal else "y")

In [ ]:
if not filtered.empty:
    monthly_sales = filtered.groupby("month")["gross_sales"].sum()
    monthly_sales.index = pd.PeriodIndex(monthly_sales.index, freq="M")
    months = pd.period_range(monthly_sales.index.min(), monthly_sales.index.max(), freq="M")
    fig, ax = plt.subplots(figsize=(10, 4), layout="constrained")
    ax.plot(months.to_timestamp(), monthly_sales.reindex(months), "o-", color=BLUE)
    ax.set(title="Monthly Gross Sales", xlabel="Month", ylabel="Gross sales (PHP)")
    finish_chart(fig, ax, "monthly_gross_sales.png", "y")

In [ ]:
if not filtered.empty:
    sales_bars(category_sales, "Gross Sales by Product Category", "Product category",
               "gross_sales_by_category.png")
    region_sales = filtered.groupby(filtered["region_name"].fillna("Unknown"))["gross_sales"].sum()
    sales_bars(region_sales, "Gross Sales by Region", "Region", "gross_sales_by_region.png")

In [ ]:
if not filtered.empty:
    fig, ax = plt.subplots(figsize=(10, 4), layout="constrained")
    ax.hist(order_values, bins=30, color=TEAL, edgecolor="white")
    ax.set(title="Order Value Distribution", xlabel="Net order value (PHP)", ylabel="Number of orders")
    finish_chart(fig, ax, "order_value_distribution.png")

    channel_sales = filtered.groupby(filtered["sales_channel"].fillna("Unknown"))["gross_sales"].sum()
    sales_bars(channel_sales, "Gross Sales by Sales Channel", "Sales channel",
               "gross_sales_by_channel.png", horizontal=False)

In [ ]:
if not filtered.empty:
    top_products = filtered.groupby("product_name")["gross_sales"].sum().nlargest(10)
    sales_bars(top_products, "Top 10 Products by Gross Sales", "Product", "top_products.png")

## Filtered transactions

Each row is an order item. The preview shows the latest 25 matching rows. The full filtered table is available in `detail`.

In [ ]:
detail_columns = [
    "order_id", "order_date", "product_name", "category_clean", "quantity",
    "unit_price_clean", "region_name", "city", "sales_channel", "discount_pct",
    "gross_sales", "net_sales",
]
detail = filtered[detail_columns].sort_values("order_date", ascending=False).reset_index(drop=True)
print(f"Filtered item rows: {len(detail):,} | Filtered unique orders: {len(order_values):,}")
display(detail.head(25))